# Chapter 04 — SOLUTIONS: Regression Models

**Instructor file — share only after exercise session.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
np.random.seed(42)

diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = diabetes.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def score(name, y_true, y_pred):
    print(f'{name:<22}  MAE={mean_absolute_error(y_true,y_pred):.2f}  '
          f'RMSE={np.sqrt(mean_squared_error(y_true,y_pred)):.2f}  '
          f'R²={r2_score(y_true,y_pred):.3f}')

print('Setup complete!')

## Task 1 Solution: Linear Regression

In [ ]:
lr_pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
score('Linear Regression', y_test, y_pred_lr)

## Task 2 Solution: Ridge vs Lasso

In [ ]:
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=1.0))])

ridge_pipe.fit(X_train, y_train)
lasso_pipe.fit(X_train, y_train)

y_pred_ridge = ridge_pipe.predict(X_test)
y_pred_lasso = lasso_pipe.predict(X_test)

score('Ridge (alpha=1.0)', y_test, y_pred_ridge)
score('Lasso (alpha=1.0)', y_test, y_pred_lasso)

print('\nLasso coefficients (zero = feature eliminated):')
coef = pd.Series(lasso_pipe.named_steps['model'].coef_, index=diabetes.feature_names)
print(coef.round(3))
print(f'Features eliminated: {list(coef[coef==0].index)}')

## Task 3 Solution: Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
score('Random Forest', y_test, y_pred_rf)

# Feature importances
importances = pd.Series(rf.feature_importances_, index=diabetes.feature_names).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='#2ecc71', alpha=0.8)
ax.set_title('Random Forest Feature Importances (Diabetes)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()
print(f'Most important feature: {importances.idxmax()} (bmi = body mass index)')

## Task 4 Solution: Model Comparison

In [ ]:
models = {
    'Linear Regression': lr_pipe,
    'Ridge (alpha=1.0)':  ridge_pipe,
    'Lasso (alpha=1.0)':  lasso_pipe,
    'Random Forest':     rf,
}

print('5-Fold Cross-Validation (R²):')
print(f'{"Model":<22}  {"Mean R²":>10}  {"Std":>6}')
print('-' * 44)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    cv_results[name] = scores
    print(f'{name:<22}  {scores.mean():>10.4f}  ±{scores.std():.4f}')

best = max(cv_results, key=lambda k: cv_results[k].mean())
print(f'\n→ Best model by cross-validation: {best}')

## Bonus Solution: Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, preds, color) in zip(axes, [
    ('Linear Regression', y_pred_lr, '#3498db'),
    ('Random Forest', y_pred_rf, '#2ecc71'),
]):
    ax.scatter(y_test, preds, alpha=0.5, s=30, color=color)
    mn, mx = min(y_test.min(), preds.min()), max(y_test.max(), preds.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name} (R²={r2_score(y_test,preds):.3f})')
    ax.legend()
plt.suptitle('Predicted vs Actual: Closer to red line = better', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()